# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will prioritize content that is both stale and visible. Staleness is relevant to the refresh/maintenance decision, while impressions measure how much search exposure the content currently receives.

Before scoring, I will check whether these two signals show a useful directional pattern. I will use bucket tables with the number of observations (n) and give each signal a simple verdict.

My baseline rule will give higher priority to pages that have more search exposure and have not been updated recently.

The rule is intended as decision support, not as proof that a page needs a specific action. A high score means the page is worth reviewing first.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
data = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
),

performance AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_90d
    FROM {TABLES['fact_daily']} f
    CROSS JOIN bounds b
    WHERE f.report_date > b.end_date - INTERVAL 90 DAY
    GROUP BY
        f.client_hash_id,
        f.content_hash_id
)

SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.impressions_90d,

    c.content_updated_date,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        b.end_date
    ) AS days_since_last_update,

    c.content_type,
    c.word_count,
    c.search_volume,
    c.is_published,
    c.is_deleted

FROM performance p

CROSS JOIN bounds b

LEFT JOIN {TABLES['dim_content']} c
    ON p.client_hash_id = c.client_hash_id
    AND p.content_hash_id = c.content_hash_id

WHERE
    c.is_deleted = FALSE
    AND c.is_published = TRUE
    AND p.impressions_90d > 0
    AND c.content_updated_date IS NOT NULL

""").df()

print(f"Baseline rows: {len(data):,}")
print()
print(data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.